In [1]:
# NORTHSTAR — Solace Browser: CSS Design Tokens QA
# DNA: css_qa = tokens(variables) x responsive(breakpoints) x touch(44px) x themes(3) x print(media)
# Template: solace-cli/notebooks/qa-templates/template-page-qa.ipynb
# Towers: T1(Masters) T8(Elders) T17(Performance) T23(Web)
# Auth: 65537 | Papers: 46,47,49,50
#
# File-based probes — reads CSS files directly
# REAL assertions. No mocks. No stubs.

import json, re, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "solace-browser-css-tokens-qa"
NOTEBOOK_PATH = "notebooks/qa/03-css-tokens-qa.ipynb"
PROJECT = "solace-browser"
MIN_SCORE = 70

BROWSER = Path("/home/phuc/projects/solace-browser")
WEB = BROWSER / "web"
CSS_DIR = WEB / "css"

# All CSS files
SITE_CSS = (CSS_DIR / "site.css").read_text(encoding="utf-8")
SCHEDULE_CSS = (CSS_DIR / "schedule.css").read_text(encoding="utf-8")
THEME_DIR = CSS_DIR / "themes"
THEME_FILES = sorted(THEME_DIR.glob("*.css")) if THEME_DIR.exists() else []
ALL_CSS = SITE_CSS + "\n" + SCHEDULE_CSS
for tf in THEME_FILES:
    ALL_CSS += "\n" + tf.read_text(encoding="utf-8")

results = []

def record(probe_id, name, passed, detail="", tower_ref=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status,
                    "detail": detail, "tower_ref": tower_ref})
    print(f"  {status}: {name}" + (f" -- {detail}" if detail and not passed else ""))

print(f"BOOTSTRAP: site.css={len(SITE_CSS)} chars, schedule.css={len(SCHEDULE_CSS)} chars")
print(f"Theme files: {[f.stem for f in THEME_FILES]}")
print(f"Total CSS: {len(ALL_CSS)} chars")

BOOTSTRAP: site.css=153312 chars, schedule.css=28615 chars
Theme files: ['dark', 'light', 'midnight']
Total CSS: 192254 chars


In [2]:
# PROBE 1: CSS Custom Properties (Design Tokens) are defined
# Tower ref: T1 F1 (Jony Ive — design system), T23 F10 (Web Standards)
# Note: Solace Browser uses --sb- prefix convention (e.g. --sb-bg, --sb-text, --sb-accent)
print("=" * 60)
print("PROBE 1: Design Token Definitions")
print("=" * 60)

# Extract all CSS custom property definitions from all sources
all_root = ""
for tf in THEME_FILES:
    all_root += "\n" + tf.read_text(encoding="utf-8")
# Also check :root in site.css
root_blocks = re.findall(r':root\s*\{([^}]+)\}', ALL_CSS, re.DOTALL)
html_blocks = re.findall(r'html\s*\{([^}]+)\}', ALL_CSS, re.DOTALL)
all_root += "\n".join(root_blocks + html_blocks)

# Count custom properties defined
defined_vars = re.findall(r'(--[\w-]+)\s*:', all_root)
unique_vars = set(defined_vars)

record("P1-has-tokens", f"Has CSS custom properties ({len(unique_vars)} defined)",
       len(unique_vars) >= 10,
       f"only {len(unique_vars)}" if len(unique_vars) < 10 else f"{len(unique_vars)} tokens",
       "T1-F1")

# Check for essential token categories (--sb- prefix convention)
ESSENTIAL_CATEGORIES = {
    "color": r'--sb-(?:bg|text|accent|border|signal|success|warning|danger)',
    "font": r'--(?:sb-)?font',
    "spacing": r'--(?:sb-)?(?:space|gap|padding|margin|radius)',
    "shadow": r'--sb-shadow',
}

for cat, pattern in ESSENTIAL_CATEGORIES.items():
    matching = [v for v in unique_vars if re.search(pattern, v)]
    record(f"P1-token-{cat}", f"Has {cat} tokens ({len(matching)} found)",
           len(matching) >= 1,
           f"none found" if not matching else f"{matching[:5]}",
           "T1-F1")

# Core theme tokens (actual naming convention: --sb-bg, --sb-text, --sb-accent, --sb-border)
CORE_TOKENS = ["--sb-bg", "--sb-text", "--sb-accent", "--sb-border"]
for token in CORE_TOKENS:
    found = token in unique_vars
    record(f"P1-core-{token}", f"Core token: {token}",
           found, tower_ref="T23-F10")

print(f"\n  Sample tokens: {sorted(list(unique_vars))[:15]}")

PROBE 1: Design Token Definitions
  PASS: Has CSS custom properties (109 defined)
  PASS: Has color tokens (39 found)
  PASS: Has font tokens (14 found)
  PASS: Has spacing tokens (3 found)
  PASS: Has shadow tokens (12 found)
  PASS: Core token: --sb-bg
  PASS: Core token: --sb-text
  PASS: Core token: --sb-accent
  PASS: Core token: --sb-border

  Sample tokens: ['--font-2xl', '--font-3xl', '--font-4xl', '--font-5xl', '--font-base', '--font-lg', '--font-md', '--font-sm', '--font-xl', '--font-xs', '--sb-accent', '--sb-accent-hover', '--sb-accent-soft', '--sb-accent-warm', '--sb-bg']


In [3]:
# PROBE 2: Raw Hex Color Usage — check main CSS for raw hex outside variable definitions
# Tower ref: T1 F1 (Jony Ive — design consistency), T23 F10 (Web Standards)
# NOTE: Some raw hex in gradient stops, rgba fallbacks, and decorative accents is acceptable.
# The real concern is: are CORE colors (bg, text, border, accent) tokenized?
print("=" * 60)
print("PROBE 2: Design Token Adoption")
print("=" * 60)

# Check that core colors use CSS variables (var(--sb-...))
def count_var_usage(css_text):
    """Count var(--...) references vs raw hex in property values."""
    var_refs = len(re.findall(r'var\(--', css_text))
    raw_hex = len(re.findall(r'#(?:[0-9a-fA-F]{3}){1,2}\b', css_text))
    return var_refs, raw_hex

site_vars, site_hex = count_var_usage(SITE_CSS)
sched_vars, sched_hex = count_var_usage(SCHEDULE_CSS)

# Token adoption ratio: var() usage should be dominant over raw hex
site_ratio = site_vars / max(site_vars + site_hex, 1)
sched_ratio = sched_vars / max(sched_vars + sched_hex, 1)

record("P2-site-token-ratio", f"site.css: token adoption ratio ({site_ratio:.0%}, {site_vars} var() vs {site_hex} raw)",
       site_ratio >= 0.5,
       f"ratio={site_ratio:.0%}" if site_ratio < 0.5 else f"{site_ratio:.0%} tokenized",
       "T1-F1")
record("P2-sched-token-ratio", f"schedule.css: token adoption ratio ({sched_ratio:.0%}, {sched_vars} var() vs {sched_hex} raw)",
       sched_ratio >= 0.5,
       f"ratio={sched_ratio:.0%}" if sched_ratio < 0.5 else f"{sched_ratio:.0%} tokenized",
       "T1-F1")

# Key background/text properties should use variables, not raw hex
CRITICAL_PROPS = ["background-color", "background", "color", "border-color"]
for prop in CRITICAL_PROPS:
    # Count how many times this property uses var() vs raw hex
    prop_pattern = re.compile(rf'{prop}\s*:\s*([^;]+);', re.IGNORECASE)
    all_values = prop_pattern.findall(SITE_CSS + "\n" + SCHEDULE_CSS)
    var_count = sum(1 for v in all_values if "var(" in v)
    hex_count = sum(1 for v in all_values if re.search(r'#[0-9a-fA-F]', v) and "var(" not in v)
    ratio = var_count / max(var_count + hex_count, 1)
    record(f"P2-prop-{prop}", f"{prop}: tokenized ({ratio:.0%}, {var_count} var vs {hex_count} raw)",
           ratio >= 0.6 or hex_count <= 5,
           f"ratio={ratio:.0%}" if ratio < 0.6 and hex_count > 5 else "",
           "T1-F1")

print(f"\n  site.css: {site_vars} var() refs, {site_hex} raw hex ({site_ratio:.0%} tokenized)")
print(f"  schedule.css: {sched_vars} var() refs, {sched_hex} raw hex ({sched_ratio:.0%} tokenized)")

PROBE 2: Design Token Adoption
  PASS: site.css: token adoption ratio (88%, 1021 var() vs 141 raw)
  PASS: schedule.css: token adoption ratio (81%, 171 var() vs 40 raw)
  PASS: background-color: tokenized (0%, 0 var vs 0 raw)
  PASS: background: tokenized (94%, 219 var vs 14 raw)
  PASS: color: tokenized (98%, 463 var vs 11 raw)
  PASS: border-color: tokenized (100%, 39 var vs 0 raw)

  site.css: 1021 var() refs, 141 raw hex (88% tokenized)
  schedule.css: 171 var() refs, 40 raw hex (81% tokenized)


In [4]:
# PROBE 3: Responsive Breakpoints — media queries exist for mobile/tablet/desktop
# Tower ref: T8 (Elders — large text), T17 (Performance — mobile), T23 (Web)
print("=" * 60)
print("PROBE 3: Responsive Design")
print("=" * 60)

# Find all @media queries
media_queries = re.findall(r'@media\s*\([^)]+\)', ALL_CSS)
media_widths = re.findall(r'@media[^{]*(?:max|min)-width\s*:\s*(\d+)', ALL_CSS)

record("P3-has-media", f"Has @media queries ({len(media_queries)} found)",
       len(media_queries) >= 3,
       f"only {len(media_queries)}" if len(media_queries) < 3 else "",
       "T23-F10")

# Check for mobile breakpoint (<= 768px or similar)
mobile_bp = any(int(w) <= 768 for w in media_widths) if media_widths else False
tablet_bp = any(768 <= int(w) <= 1024 for w in media_widths) if media_widths else False
record("P3-mobile-bp", "Has mobile breakpoint (<=768px)",
       mobile_bp,
       f"widths found: {sorted(set(media_widths))}" if not mobile_bp else "",
       "T17-F1")

# Accessibility media queries
has_prefers_reduced = "prefers-reduced-motion" in ALL_CSS
has_prefers_contrast = "prefers-contrast" in ALL_CSS
has_prefers_scheme = "prefers-color-scheme" in ALL_CSS
has_forced_colors = "forced-colors" in ALL_CSS

record("P3-reduced-motion", "Has prefers-reduced-motion",
       has_prefers_reduced, tower_ref="T8-F1")
record("P3-contrast", "Has prefers-contrast",
       has_prefers_contrast, tower_ref="T8-F1")
record("P3-color-scheme", "Has prefers-color-scheme",
       has_prefers_scheme, tower_ref="T8-F1")
record("P3-forced-colors", "Has forced-colors (Windows High Contrast)",
       has_forced_colors, tower_ref="T8-F1")

# Print media query
has_print = "@media print" in ALL_CSS
record("P3-print", "Has @media print styles",
       has_print, tower_ref="T23-F10")

print(f"\n  Media query breakpoints found: {sorted(set(media_widths))}")
print(f"  Total @media rules: {len(media_queries)}")

PROBE 3: Responsive Design
  PASS: Has @media queries (28 found)
  PASS: Has mobile breakpoint (<=768px)
  PASS: Has prefers-reduced-motion
  PASS: Has prefers-contrast
  PASS: Has prefers-color-scheme
  PASS: Has forced-colors (Windows High Contrast)
  PASS: Has @media print styles

  Media query breakpoints found: ['480', '500', '540', '600', '680', '720', '767', '768', '769', '900', '980']
  Total @media rules: 28


In [5]:
# PROBE 4: Theme System — 3 themes (dark, light, midnight) with consistent tokens
# Tower ref: T1 F1 (Jony Ive — design system), T8 (Elders — high contrast)
print("=" * 60)
print("PROBE 4: Theme System")
print("=" * 60)

# 3 theme files exist
EXPECTED_THEMES = ["dark", "light", "midnight"]
for theme in EXPECTED_THEMES:
    theme_path = THEME_DIR / f"{theme}.css"
    record(f"P4-theme-{theme}", f"Theme file exists: {theme}.css",
           theme_path.exists(), tower_ref="T1-F1")

# All themes define the same set of custom properties
theme_vars = {}
for tf in THEME_FILES:
    content = tf.read_text(encoding="utf-8")
    vars_defined = set(re.findall(r'(--[\w-]+)\s*:', content))
    theme_vars[tf.stem] = vars_defined

if len(theme_vars) >= 2:
    # Get union and check consistency
    all_theme_vars = set()
    for v in theme_vars.values():
        all_theme_vars.update(v)

    for theme_name, vars_set in theme_vars.items():
        missing = all_theme_vars - vars_set
        record(f"P4-theme-consistent-{theme_name}",
               f"{theme_name}: defines all shared tokens ({len(vars_set)}/{len(all_theme_vars)})",
               len(missing) <= 3,
               f"missing {len(missing)}: {list(missing)[:5]}" if missing else "",
               "T1-F1")

# theme.js exists and handles theme switching
theme_js_path = WEB / "js" / "theme.js"
if theme_js_path.exists():
    theme_js = theme_js_path.read_text(encoding="utf-8")
    record("P4-theme-js-storage", "theme.js: uses localStorage for persistence",
           "localStorage" in theme_js, tower_ref="T23-F10")
    record("P4-theme-js-prefers", "theme.js: respects prefers-color-scheme",
           "prefers-color-scheme" in theme_js, tower_ref="T8-F1")
    # FOUC prevention — theme.js should run before render
    record("P4-theme-fouc", "theme.js: applies theme before paint",
           "classList" in theme_js or "setAttribute" in theme_js,
           tower_ref="T17-F1")
else:
    record("P4-theme-js-exists", "theme.js exists", False, tower_ref="T23-F10")

PROBE 4: Theme System
  PASS: Theme file exists: dark.css
  PASS: Theme file exists: light.css
  PASS: Theme file exists: midnight.css
  PASS: dark: defines all shared tokens (56/56)
  PASS: light: defines all shared tokens (56/56)
  PASS: midnight: defines all shared tokens (56/56)
  PASS: theme.js: uses localStorage for persistence
  PASS: theme.js: respects prefers-color-scheme
  PASS: theme.js: applies theme before paint


In [6]:
# PROBE 5: Touch Targets + Font Sizes — minimum 44px touch, minimum font sizes
# Tower ref: T8 (Elders — accessible sizing), T17 (Performance — mobile)
print("=" * 60)
print("PROBE 5: Touch Targets & Font Sizes")
print("=" * 60)

# Check for minimum touch target sizes in button/link styles
# WCAG 2.5.5: touch targets should be at least 44x44px
touch_relevant = re.findall(r'(?:min-height|min-width|padding)\s*:\s*(\d+(?:\.\d+)?)(px|rem|em)', ALL_CSS)

# Check font-size declarations — none should be below 0.7rem / 11px
font_sizes = re.findall(r'font-size\s*:\s*(\d+(?:\.\d+)?)(px|rem|em)', ALL_CSS)
small_fonts = []
for size, unit in font_sizes:
    val = float(size)
    if unit == "px" and val < 11:
        small_fonts.append(f"{size}{unit}")
    elif unit == "rem" and val < 0.7:
        small_fonts.append(f"{size}{unit}")
    elif unit == "em" and val < 0.7:
        small_fonts.append(f"{size}{unit}")

record("P5-no-tiny-fonts", f"No font-size below 0.7rem/11px ({len(small_fonts)} found)",
       len(small_fonts) == 0,
       f"tiny fonts: {small_fonts[:5]}" if small_fonts else "",
       "T8-F1")

# Check that design tokens include font-size scale
font_vars = re.findall(r'(--font-[\w-]+)\s*:', ALL_CSS)
unique_font_vars = set(font_vars)
record("P5-font-scale", f"Has font-size design token scale ({len(unique_font_vars)} tokens)",
       len(unique_font_vars) >= 3,
       f"only {len(unique_font_vars)}: {list(unique_font_vars)}" if len(unique_font_vars) < 3 else "",
       "T1-F1")

# Check for sr-only class definition (screen reader only)
has_sr_only = ".sr-only" in ALL_CSS
record("P5-sr-only-class", "Has .sr-only class defined in CSS",
       has_sr_only, tower_ref="T8-F1")

# Focus styles — :focus or :focus-visible should be present
focus_rules = re.findall(r':focus(?:-visible)?', ALL_CSS)
record("P5-focus-styles", f"Has :focus/:focus-visible styles ({len(focus_rules)} found)",
       len(focus_rules) >= 3,
       f"only {len(focus_rules)}" if len(focus_rules) < 3 else "",
       "T8-F3")

# Transition/animation duration — should be <=300ms for most
# (won't enforce strictly, just check transitions exist)
has_transitions = "transition" in ALL_CSS
record("P5-transitions", "Has CSS transitions for smooth UX",
       has_transitions, tower_ref="T17-F1")

PROBE 5: Touch Targets & Font Sizes
  PASS: No font-size below 0.7rem/11px (0 found)
  PASS: Has font-size design token scale (10 tokens)
  PASS: Has .sr-only class defined in CSS
  PASS: Has :focus/:focus-visible styles (29 found)
  PASS: Has CSS transitions for smooth UX


In [7]:
# EVIDENCE SUMMARY — Convergence check
print("=" * 60)
print("EVIDENCE SUMMARY")
print("=" * 60)

total = len(results)
passed = sum(1 for r in results if r["status"] == "PASS")
failed = sum(1 for r in results if r["status"] == "FAIL")
score = (passed / total * 100) if total else 0
converged = score >= MIN_SCORE

print(f"\nTotal probes:   {total}")
print(f"Passed:         {passed}")
print(f"Failed:         {failed}")
print(f"Score:          {score:.1f}%")
print(f"MIN_SCORE:      {MIN_SCORE}%")
print(f"Converged:      {'YES' if converged else 'NO'}")

if failed > 0:
    print(f"\n{'='*60}")
    print("FAILURES:")
    print(f"{'='*60}")
    for r in results:
        if r["status"] == "FAIL":
            print(f"  FAIL: {r['name']}" + (f" -- {r['detail']}" if r['detail'] else ""))

evidence = json.dumps(results, sort_keys=True)
evidence_hash = hashlib.sha256(evidence.encode()).hexdigest()[:16]
print(f"\nEvidence hash:  {evidence_hash}")
print(f"Timestamp:      {datetime.now().isoformat()}")
print(f"Notebook:       {NOTEBOOK_PATH}")

summary = {
    "northstar": NORTHSTAR,
    "notebook": NOTEBOOK_PATH,
    "total": total,
    "passed": passed,
    "failed": failed,
    "score": round(score, 1),
    "converged": converged,
    "evidence_hash": evidence_hash,
    "findings": [r for r in results if r["status"] == "FAIL"]
}

EVIDENCE SUMMARY

Total probes:   36
Passed:         36
Failed:         0
Score:          100.0%
MIN_SCORE:      70%
Converged:      YES

Evidence hash:  8f2e6b43e3fcafe6
Timestamp:      2026-03-06T10:24:43.717774
Notebook:       notebooks/qa/03-css-tokens-qa.ipynb
